In [2]:
import torch 
import torch.nn as nn 
from torch.nn import functional as F 
from dataclasses import dataclass

In [3]:
@dataclass
class GPTConfig:
    block_size  = 1024#256   # 1024, B==> block size  
    vocab_size  = 50257#65    # 50257  ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer     = 12#6     # 12 
    n_head      = 12#6     # 12 
    n_emb       = 768#384   # 768   C==> embedding dim 

In [4]:
x = torch.randn(4,GPTConfig.block_size,GPTConfig.n_emb) 
x.shape

torch.Size([4, 1024, 768])

In [5]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch 
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection 
        self.c_proj.NANOGPT_SCALE_INIT  = 1
        
        self.n_head = config.n_head
        self.n_emb  = config.n_emb
    
    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim 
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size" 
        # C (number of channels)= nh * hs 
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        y       = F.scaled_dot_product_attention(q,k,v,is_causal=True) #(B,n_head,T,head_size) # Flash attention==> its faster,cleaner,and scale better than Head object that created (reference link:-https://github.com/Vampaxx/Language_Model/blob/main/GPT_from_scratch/07_Adding_new_parameters.py )
        y       = y.transpose(1,2).contiguous().view(B,T,C) 
        #output projection 
        y       = self.c_proj(y)       
        return y

In [33]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_emb % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_emb, 3 * config.n_emb)
        # output projection
        self.c_proj = nn.Linear(config.n_emb, config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_emb = config.n_emb

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh is "number of heads", hs is "head size", and C (number of channels) = nh * hs
        # e.g. in GPT-2 (124M), n_head=12, hs=64, so nh*hs=C=768 channels in the Transformer
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True) # flash attention
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

In [34]:
for i,wei in CausalSelfAttention(GPTConfig).state_dict().items():
    print(i,wei.shape)

c_attn.weight torch.Size([2304, 768])
c_attn.bias torch.Size([2304])
c_proj.weight torch.Size([768, 768])
c_proj.bias torch.Size([768])


In [35]:
GPTConfig.n_emb

768

🤔 Why `y.contigious()`is this needed?

- In PyTorch, certain tensor operations (like .transpose() or .permute()) don't move data — they just return a view with a new stride, meaning:

    - The tensor looks different when indexed,
    - But the underlying memory layout hasn't changed.

| `.contiguous()` does...                          | Why it's needed                                                        |
| ------------------------------------------------ | ---------------------------------------------------------------------- |
| Makes tensor memory layout **continuous**        | Some operations like `.view()` or CUDA kernels need contiguous memory  |
| Copies tensor data into **new memory** if needed | `.transpose()` or `.permute()` only change the view, not memory layout |
| Used for **safety & correctness**                | Prevents runtime errors or silent bugs due to incompatible strides     |

- Calls .view() on a non-contiguous tensor — can crash


🧩 Why `self.c_proj.NANOGPT_SCALE_INIT  = 1`?

- In GPT-style transformers, it's common to:

    - Use smaller initial weights on projection layers (like c_proj)
    - This helps prevent instability, especially with residual connections

This flag is just a way to mark those layers.

| Attribute                                        | Meaning                                                                |
| ------------------------------------------------ | ---------------------------------------------------------------------- |
| `NANOGPT_SCALE_INIT`                             | Marks this layer for special weight scaling during model initialization|
| is it required?                                  | No,but its helpful for model statbility in deep models                 |
| Where is it used?                                | Usually in post in weight scaling logic                                |


In [6]:
self_attention = CasualSelfAttention(GPTConfig)
out = self_attention.forward(x)
out.shape

torch.Size([4, 1024, 768])

In [23]:
for i,wei in CasualSelfAttention(GPTConfig).state_dict().items():
    print(i,wei.shape)

c_attn.weight torch.Size([2304, 768])
c_attn.bias torch.Size([2304])
c_proj.weight torch.Size([768, 768])
c_proj.bias torch.Size([768])


- self.c_attn (Combined Attention Projection):

    - self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        - This linear layer -->  generating the Query (Q), Key (K), and Value (V) projections for all attention heads in a single batch operation.

        - Instead of having three separate nn.Linear layers for Q, K, and V, it's a common optimization to use one larger linear layer that outputs a tensor three times the size of the embedding dimension. This output is then typically split into Q, K, and V components.
        - So, c_attn takes the input embeddings (config.n_embd) and projects them into a combined space that will later be divided into Q, K, and V for the attention calculation.

- self.c_proj (Output Projection):

    - self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    - After the attention scores are calculated and the **Value vectors are weighted and summed across all attention heads**, the results from these multiple heads are typically concatenated.

    - This concatenated output needs to be projected back to the original embedding dimension (config.n_embd) so that it can be passed to subsequent layers in the model (e.g., a feed-forward network).

    - c_proj performs this final linear transformation, effectively combining the information from all attention heads and mapping it back to the expected dimensionality. The NANOGPT_SCALE_INIT = 1 is a specific initialization scaling factor used in some implementations (like NanoGPT) for better training stability.

In [7]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT  = 1
    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x         

In [22]:
for i,wei in MLP(GPTConfig).state_dict().items():
    print(i,wei.shape)

c_fc.weight torch.Size([3072, 768])
c_fc.bias torch.Size([3072])
c_proj.weight torch.Size([768, 3072])
c_proj.bias torch.Size([768])


In [21]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x 

In [19]:
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte     = nn.Embedding(self.config.vocab_size,self.config.n_emb),           # token embeding
            wpe     = nn.Embedding(self.config.block_size,self.config.n_emb),           # position embedding 
            h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads  
            ln_f    = nn.LayerNorm(config.n_emb)
        ))
        self.lm_head= nn.Linear(config.n_emb,config.vocab_size)        
    def hey(self):
        pass

In [20]:
for i,wei in Block(GPTConfig).state_dict().items():
    print(i,wei.shape)

ln_1.weight torch.Size([768])
ln_1.bias torch.Size([768])
attn.c_attn.weight torch.Size([2304, 768])
attn.c_attn.bias torch.Size([2304])
attn.c_proj.weight torch.Size([768, 768])
attn.c_proj.bias torch.Size([768])
ln_2.weight torch.Size([768])
ln_2.bias torch.Size([768])
mlp.c_fc.weight torch.Size([3072, 768])
mlp.c_fc.bias torch.Size([3072])
mlp.c_proj.weight torch.Size([768, 3072])
mlp.c_proj.bias torch.Size([768])


In [ ]:
model = GPT(GPTConfig)
for i,wei in model.state_dict().items():
    print(i,wei.shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln1.weight torch.Size([768])
transformer.h.0.ln1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([2304, 768])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln2.weight torch.Size([768])
transformer.h.0.ln2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln1.weight torch.Size([768])
transformer.h.1.ln1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([2304, 768])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias torch.

In [ ]:
dasd

NameError: name 'dasd' is not defined

In [ ]:
B = 4 
T = 256
n_emb = 384 
n_head = 6 
head_size = n_emb // n_head
head_size

64

In [ ]:
c_attn  = nn.Linear(n_emb,3*n_emb)    # c_attn.w  =   384,3*384 
x       = torch.randn(B,T,384)  #           = 4, 32,64
                                # x * c_attn.W.T == 4
qkv     = c_attn(x)             # ==> 4,32,192 
q,k,v   = qkv.split(384,dim=2)   # 3 * (4,32,64)

In [ ]:
attn = q @k.transpose(2,1)
attn.shape

torch.Size([4, 256, 256])

In [ ]:
c_attn.weight.shape

torch.Size([1152, 384])

In [ ]:
qkv.shape

torch.Size([4, 256, 1152])

In [ ]:
from math import sqrt 

In [ ]:
(q @ k.transpose(2,1)) * (1/math.sqrt())

torch.Size([4, 256, 256])

In [ ]:
q_ = q.view(B,T,n_head,n_emb//n_head) # B,T,n_h,n_emb // n_heads
q_.shape

torch.Size([4, 256, 6, 64])

In [ ]:
q_.size()

torch.Size([4, 256, 6, 64])

In [ ]:
q_.size(-1)

64

In [ ]:
batch_size  = 4     # B 
block_size  = 128    # T
n_emd       = 384    # C 
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 500
learning_rate   = 1e-3
max_iters       = 6000 
eval_iter       = 200
n_layers        = 6 
n_heads         = 6
dropout         = 0.2
vocab_size      = 65 

head_size = n_emd // n_heads

In [ ]:
x = torch.randn(batch_size,block_size,n_emd)
print(batch_size,block_size,n_emb)

4 128 384


In [ ]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

        self.dropout    = nn.Dropout(dropout)
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        wei     = self.dropout(wei)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [ ]:
head_list = nn.ModuleList([Head(head_size) for _ in range(n_layers)])

In [ ]:
a = torch.cat([head(x) for head in head_list])

In [ ]:
a.shape

torch.Size([24, 128, 64])

In [ ]:
for i in range(6):
    print(([head(x) for head in head_list][i]).shape)

torch.Size([4, 128, 64])
torch.Size([4, 128, 64])
torch.Size([4, 128, 64])
torch.Size([4, 128, 64])
torch.Size([4, 128, 64])
torch.Size([4, 128, 64])


In [ ]:
a = torch.randint(12,24,(2,6))
b = torch.randint(30,42,(1,6))

In [ ]:
a,b

(tensor([[14, 16, 19, 14, 23, 18],
         [19, 16, 17, 15, 18, 23]]),
 tensor([[40, 35, 33, 36, 31, 35]]))

In [ ]:
torch.cat((a,b)).shape

torch.Size([3, 6])

In [ ]:
block_ = Block(GPTConfig)
block_out = block_(x)

In [ ]:
from transformers import GPT2LMHeadModel

c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_type  = "gpt2"
config_args = {
    "gpt2"          : dict(n_layer  = 12, n_head = 12,n_emb = 768),
    "gpt2-medium"   : dict(n_layer  = 24, n_head = 16,n_emb = 1024),
    "gpt2-large"    : dict(n_layer  = 36, n_head = 20,n_emb = 1280),
    "gpt2-xl"       : dict(n_layer  = 48, n_head = 25,n_emb = 1600)
}[model_type]
config_args['vocab_size']   = 50257
config_args['block_size']   = 1024
config                      = GPTConfig(**config_args)
model                       = GPT(config)
sd                          = model.state_dict()
sd_keys                     = sd.keys()
sd_keys                     = [k for k in sd_keys if not k.endswith(".attn.bias")]

model_hf                    = GPT2LMHeadModel.from_pretrained(model_type)
sd_hf                       = model_hf.state_dict()


In [ ]:
config_args['vocab_size']   = 50257
config_args['block_size']   = 1024

In [ ]:
config_args

{'n_layer': 12,
 'n_head': 12,
 'n_emb': 768,
 'vocab_size': 50257,
 'block_size': 1024}